In [ ]:
# google colabにzstdがないので、DL
!sudo apt-get update && sudo apt-get install -y zstd

In [ ]:
# ollamaのインストール
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!uv pip install lightrag-hku nest-asyncio pyvis networkx

In [ ]:
import asyncio
import glob
import os
import subprocess
import time

import IPython
import nest_asyncio
import networkx as nx
from google.colab import userdata
from tenacity import retry, wait_fixed, stop_after_attempt
from lightrag import LightRAG, QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
# from lightrag.llm.ollama import ollama_model_complete, ollama_embed # インデックス構築にOllamaを使用する場合
from lightrag.llm.gemini import gemini_model_complete # インデックス構築のLLMはGemini
from lightrag.llm.ollama import ollama_embed          # Embeddings化はOllama
from lightrag.utils import EmbeddingFunc
from pyvis.network import Network


In [ ]:
# ollamaサーバーをバックグランドで起動
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # 起動を少し待つ

In [ ]:
# 生成用LLMモデルダウンロード
# !ollama pull qwen2.5:3b

# 埋め込み(Embedding)モデルをダウンロード
# !ollama pull nomic-embed-text # 768次元のモデル
!ollama pull bge-m3 # 1024次元のモデル


In [ ]:
print("DL DONE!!!!!!!!!")

In [ ]:
# グラフデータの保存先作成
WORKING_DIR = "./rag_storage"

if not os.path.exists(WORKING_DIR):
    os.makedirs(WORKING_DIR)

In [ ]:
README_DIR = "./readmes"
if not os.path.exists(README_DIR):
    os.makedirs(README_DIR)

In [ ]:
# geminiの設定
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
gemini_model_name = "gemini-2.5-flash"
# gemini_model_name = "gemini-3.5-flash"


## 1. LightRAGの設定

In [ ]:
# 非同期処理のコンフリクト対策
nest_asyncio.apply()

In [ ]:
# gemini_model_completeにretry機能をつける
# gemini apiの1分間のレート制限対策

@retry(
    wait = wait_fixed(60),
    # 最大3回までリトライを試みる
    stop = stop_after_attempt(3),
    reraise = True
)
async def gemini_model_complete_with_retry(prompt, system_prompt=None, history_messages=None, **kwargs):

    # 無料枠のレート制限を考慮し、
    # マージンのため待機時間を入れる
    
    await asyncio.sleep(20)

    return await gemini_model_complete(
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs
    )

In [ ]:
# embeddingはOllamaでローカル実行し、gemini apiの利用回数を抑える。
# インデックス化はgemini apiで行う。

kg_rag = LightRAG(
    workspace="hybrid_workspace",
    working_dir=WORKING_DIR,
    # llm_model_func=gemini_model_complete,
    llm_model_func=gemini_model_complete_with_retry, # gemini apiのレート制限対策で待機をいれたバージョン
    llm_model_name=gemini_model_name,

    # embeddingはOllmaでローカルで行う。gemini apiの無料枠のレート制限対策
    embedding_func=EmbeddingFunc(
        # embedding_dim=768, # nomic-embed-textの出力次元数
        embedding_dim = 1024, # bge-m3の出力次元数
        # max_token_size=8192,
        max_token_size=2048, # embeddingsのタイムアウトを避けるため最大の8192ではなく小さく処理する
        # func=lambda texts: ollama_embed(texts, embed_model="nomic-embed-text")
        func = lambda texts: ollama_embed(texts, embed_model="bge-m3")

    ),

    # Ollmaやgeminiの衝突防止などのために、同時実行数の制限
    llm_model_max_async=1,
    embedding_func_max_async=1,
)

In [ ]:
# ナレッジグラフの対象となるreadmeファイルのパスの読み込み

list_readme_files = glob.glob("./readmes/*.md")
list_readme_files

## 2. インデックス構築

In [ ]:

# 非同期でインデックス処理を行う関数を定義
async def build_graph_index():

    if not list_readme_files:
        print("readmeをアップしてください")
        return

    await kg_rag.initialize_storages()      # ストレージの初期化
    await initialize_pipeline_status()      # パイプラインの初期化

    # readmeを読み込んで、グラフ作成
    for idx, file_path in enumerate(list_readme_files, start=1):
            print(f"[{idx}/{len(list_readme_files)}] グラフ構築中: {file_path}")

            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()

            try:
                await kg_rag.ainsert(content)
                print(" -> 成功")
            except Exception as e:
                print(f" -> エラーが発生: {e}")
                raise

            # 非同期関数の中であるため、time.sleepではなく、asyncio.sleep
            await asyncio.sleep(3)

    print("\nデータをディスク（rag_storage）に書き出し中...")
    await kg_rag.finalize_storages()

    print("知識グラフインデックスの構築完了")


In [ ]:
await build_graph_index()

## 3. クエリで検索

In [ ]:
async def answering(query):

    await kg_rag.initialize_storages()      # ストレージの初期化
    await initialize_pipeline_status()      # パイプラインの初期化

    answer = await kg_rag.aquery(
        query,
        param=QueryParam(mode="hybrid") # ローカル情報とグローバル情報を組み合わせる設定
    )

    print("【質問】:", query)
    print("-" * 50)
    print("【回答】:\n", answer)


In [ ]:
# 動作確認
query = "ポートフォリオ全体を通じて、共通する開発テーマや使われている主な技術は何ですか？"

await answering(query)


## 4. 通常RAGとの比較

In [ ]:
# query = "太陽光発電量予測で使ったモデルは？"
# query = "LightGBMを使ったプロジェクトを教えて"
# query = "Dockerを利用したプロジェクトは？"
# query = "ポートフォリオ全体を通じて、共通しているツールは？"
# query = "ポートフォリオのテーマの推移は？"
query = "全体を通じて、一番目立つ単語は？逆に、特定のプロジェクにしかない単語は？"

In [ ]:
await answering(query)

## 5. 作成したグラフを可視化

In [ ]:
# グラフの読み込み

graph_path = "./rag_storage/hybrid_workspace/graph_chunk_entity_relation.graphml"
G = nx.read_graphml(graph_path)

In [ ]:
net = Network(
    notebook=True, 
    dn_resources="in_line",
    height="600px",
    width="100%",
    bgcolor="#222222",
    font_color="white"
)


In [ ]:
net.from_nx(G)

In [ ]:
# # 各ノードの設定。
# サイズと、ラベルの設定

for node in net.nodes:
    node["size"] = 15
    node["label"] = node.get("id")

In [ ]:
# 物理エンジンの設定（ノードがきれいに分散するように調整）
net.toggle_physics(True)

In [ ]:
# HTMLファイルとして保存し、Colab上にインライン表示
net.show("lightrag_graph.html")
IPython.display.HTML(filename="lightrag_graph.html")

In [ ]:
# ファイルのダウンロード

from google.colab import files

files.download("lightrag_graph.html")